# Synthetic artifact sanity walkthrough

## Goal
Inspect the deterministic, non-participant synthetic artifact used by the package smoke path. This notebook is a data-contract walkthrough, not a benchmark or a neural-decoding claim.

## Setup

Run `ott data prepare --dataset synthetic --root /tmp/ott-synthetic` first. The only input below is the generated JSONL manifest and its local JSON signal fixtures. Do not point this notebook at participant data.

In [ ]:
from pathlib import Path

from openthought2text.data import load_manifest, load_json_tensor_samples

artifact_root = Path('/tmp/ott-synthetic')
manifest = load_manifest(artifact_root / 'synthetic_manifest.jsonl')
rows = load_json_tensor_samples(manifest, artifact_root)
len(rows), manifest.dataset_id, manifest.information_access.to_dict()

## Steps

The next cell summarizes shape, split, and timing metadata. It deliberately avoids target text; shapes and masks must be derived from the signal artifact rather than a reference sentence.

In [ ]:
summary = [
    {
        'sample_id': row.sample.sample_id,
        'split': row.sample.split,
        'signal_shape': tuple(row.signal.shape),
        'sampling_rate_hz': row.sample.signal.sampling_rate_hz,
    }
    for row in rows
]
summary[:5]

## Checks

Verify that the fixture contains only declared synthetic splits and finite tensors. This is a local sanity check, not numerical parity against a real dataset converter.

In [ ]:
import torch

assert {row.sample.split for row in rows} <= {'train', 'validation', 'test'}
assert all(torch.isfinite(row.signal).all() for row in rows)
assert all(row.sample.dataset_id == manifest.dataset_id for row in rows)
'synthetic artifact checks passed'

## Next Steps

For an authorized real dataset, follow `docs/templates/ADD_DATASET.md`: validate a dataset card and release bundle, create a preflight plan, derive a leakage-safe split, and save target-free full/control predictions before interpreting metrics.

Validation status: unexecuted in the repository because Jupyter/nbformat is intentionally not a default dependency. Execute with `python -m jupyter nbconvert --execute --to notebook --inplace examples/notebooks/synthetic_artifact_sanity.ipynb` in a development environment that installs Jupyter.